# House Price Prediction System
This notebook reproduces the complete workflow with real King County house-sale data: validation, exploratory data analysis, Multiple Linear Regression, residual analysis, and comparison with Random Forest. Run the cells from top to bottom in Google Colab.

In [ ]:
import os, subprocess, sys
if not os.path.exists('/content/house-price-prediction-system'):
    subprocess.run(['git', 'clone', 'https://github.com/kamrullab/house-price-prediction-system.git', '/content/house-price-prediction-system'], check=True)
os.chdir('/content/house-price-prediction-system')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project ready:', os.getcwd())

## Load and inspect the real dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.model import FEATURES, compare_models, train_and_evaluate

data = pd.read_csv('data/housing.csv')
display(data.head())
print(f'Rows: {len(data):,}')
display(data.describe(include='all').T)

## Exploratory data analysis

In [ ]:
sns.set_theme(style='whitegrid')
numeric = data[['area', 'bedrooms', 'bathrooms', 'age', 'price']]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0])
axes[0].set_title('Feature Correlation Heatmap')
sample = data.sample(min(5000, len(data)), random_state=42)
sns.scatterplot(data=sample, x='area', y='price', alpha=.35, s=22, ax=axes[1])
axes[1].set_title('Living Area and Sale Price')
plt.tight_layout(); plt.show()

## Train and evaluate Multiple Linear Regression

In [ ]:
model, metrics, comparison = train_and_evaluate(data)
display(pd.DataFrame([metrics]).rename(columns={'mae':'MAE', 'rmse':'RMSE', 'r2':'R2'}))
comparison['residual'] = comparison['actual'] - comparison['predicted']
comparison.head()

## Actual versus predicted values and residual analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.scatterplot(data=comparison, x='actual', y='predicted', alpha=.35, ax=axes[0])
limits = [comparison[['actual','predicted']].min().min(), comparison[['actual','predicted']].max().max()]
axes[0].plot(limits, limits, '--', color='red'); axes[0].set_title('Actual and Predicted Prices')
sns.scatterplot(data=comparison, x='predicted', y='residual', alpha=.35, ax=axes[1])
axes[1].axhline(0, color='red', linestyle='--'); axes[1].set_title('Residual Plot')
sns.histplot(comparison['residual'], bins=50, kde=True, ax=axes[2]); axes[2].set_title('Residual Distribution')
plt.tight_layout(); plt.show()

## Compare Linear Regression with Random Forest

In [ ]:
results = compare_models(data)
display(results.style.format({'mae':'${:,.0f}', 'rmse':'${:,.0f}', 'r2':'{:.4f}'}))

## Make one prediction

In [ ]:
example = pd.DataFrame([[1800, 3, 2, 20, '98103']], columns=FEATURES)
prediction = model.predict(example)[0]
print(f'Estimated sale price: ${prediction:,.0f}')

## Interpretation
Linear Regression provides transparent coefficients and satisfies the project proposal. Random Forest is included as a nonlinear benchmark. MAE and RMSE measure error in US dollars, while R2 describes the fraction of holdout variance explained. These historical 2014-2015 results are educational and are not current professional valuations.